# Ask My Docs RAG Walkthrough (Learning-First)

This notebook uses the reusable backend modules in `src/ask_my_docs` and walks through:

1. Ingest
2. Index
3. Retrieve
4. Rerank
5. Answer
6. Evaluate


In [ ]:
from pathlib import Path
import json
import sys

cwd = Path.cwd().resolve()
if (cwd / 'src').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'src').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError('Could not find project root with src/')

if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
from ask_my_docs.ingestion import build_chunks_from_docs, chunks_to_dataframe
from ask_my_docs.indexing import build_and_save_indexes
from ask_my_docs.pipeline import AskMyDocsEngine
from ask_my_docs.evaluation import evaluate_engine, load_eval_dataset, load_thresholds, apply_gate
from ask_my_docs.settings import SETTINGS

docs_dir = PROJECT_ROOT / 'data/docs'
index_dir = PROJECT_ROOT / 'artifacts/index'
eval_path = PROJECT_ROOT / 'data/eval/qa.yaml'
thresholds_path = PROJECT_ROOT / 'configs/eval_thresholds.yaml'

index_dir.mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'artifacts/eval').mkdir(parents=True, exist_ok=True)


## 1) Ingest


In [ ]:
chunks = build_chunks_from_docs(
    docs_dir=docs_dir,
    chunk_size_words=SETTINGS.chunk_size_words,
    chunk_overlap_words=SETTINGS.chunk_overlap_words,
)
chunks_df = chunks_to_dataframe(chunks)
print('num_chunks =', len(chunks))
chunks_df.select(['chunk_id', 'title', 'token_count']).head(10)


## 2) Index


In [ ]:
build_and_save_indexes(
    chunks_df=chunks_df,
    index_dir=index_dir,
    embedding_model=SETTINGS.embedding_model,
    reranker_model=SETTINGS.reranker_model,
)
print('index files:', sorted([p.name for p in index_dir.iterdir()]))


## 3) Retrieve + 4) Rerank


In [ ]:
engine = AskMyDocsEngine.from_index(
    index_dir=index_dir,
    embedding_model=SETTINGS.embedding_model,
    reranker_model=SETTINGS.reranker_model,
    bm25_weight=SETTINGS.bm25_weight,
    vector_weight=SETTINGS.vector_weight,
    rrf_k=SETTINGS.hybrid_rrf_k,
    candidate_pool_size=SETTINGS.candidate_pool_size,
    default_top_k=SETTINGS.default_top_k,
)

question = 'What is the SLA for invoice disputes?'
trace = engine.retriever.retrieve_with_traces(question, top_k=5)
for c in trace['top_chunks']:
    print({
        'rank': c.rank,
        'chunk_id': c.chunk_id,
        'bm25': c.bm25_score,
        'vector': c.vector_score,
        'hybrid': c.hybrid_score,
        'rerank': c.rerank_score,
    })


## 5) Answer (Citation-Enforced)


In [ ]:
response = engine.ask(question, top_k=5)
print(response.answer)


## 6) Evaluate + Gate


In [ ]:
dataset = load_eval_dataset(eval_path)
report = evaluate_engine(engine=engine, dataset=dataset, retrieval_k=5)
metrics = report['metrics']
thresholds = load_thresholds(thresholds_path)
passed, failures = apply_gate(metrics={k: float(v) for k, v in metrics.items()}, thresholds=thresholds)

print('metrics:')
for k, v in metrics.items():
    print(f'- {k}: {float(v):.4f}')

print('\ngate_passed =', passed)
if failures:
    print('failures:', failures)

(PROJECT_ROOT / 'artifacts/eval/report_notebook.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print('saved report to artifacts/eval/report_notebook.json')
